# Principal-component model in a 3-stock market

In this script we show how to use principal-component LFM to perform dimension reduction of risk
in a 3-dimensional stock market

In [1]:
from numpy import array, average, cov, eye, r_, sqrt, sum
from numpy.linalg import eigh
from pandas import read_csv, to_datetime

# r_ is numpy's shorthand fr stacking arrays row-wise.
# It's used at the end to build the residual-plus-factor matrix

In [2]:
# find the ROOT & DATA folder.

from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
DATA_DIR = ROOT / "data"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))  # lets 'from arpym...' work from any subfolder

Load Data

From database db_stocks_sp we upload the stocks values

In [5]:
# header=1 means the column names are on the files 2nd row
# usecols picks 3 stocks, pandas keeps them in the files column order
db_stocks_sp = read_csv(DATA_DIR/"db_stocks_sp.csv", parse_dates=True, date_format='mixed',
                        index_col=0, header=1, usecols=['date', 'CVC', 'AON', 'CSCO'])

# stock values
# date filter taking data from 16th-30th March 2012
# .T transposes again so v_stocks is stocks vs days (3x11)
v_stocks = array(db_stocks_sp[(db_stocks_sp.index >= to_datetime('16-03-2012', dayfirst=True))&
                              (db_stocks_sp.index <= to_datetime('30-03-2012', dayfirst=True))].values).T

# confirms which row is which stock visually
print(db_stocks_sp.columns)

Index(['AON', 'CSCO', 'CVC'], dtype='str')


1. Principal-component model in a 3-stock market

Targets, mean, covariance

In [15]:
# target variables
# [:, 1:] -> todays
# [:, :-1] -> yesterdays
x = v_stocks[:, 1:]/v_stocks[:, :-1] - 1
print('x \n', x)
print()
print('x shape : ', x.shape)
print()

# expectation
# reshape((-1, 1) -> makes the mean a 3 x 1 column, so it can be subtracted from every days column later.
e_x = average(x, axis=1).reshape((-1, 1))
print('e_x \n', e_x.round(5))
print()

# covariance
# cov(x) -> divides T-1 (not T)
cv_x = cov(x)
print('cv_x \n', cv_x.round(5))

x 
 [[ 0.00061208 -0.00652895 -0.00225972 -0.00452968  0.00641216  0.0069852
  -0.00775441  0.01131257 -0.00406802  0.00183829]
 [ 0.00549296  0.02134871 -0.00340331 -0.00634058  0.00687344  0.01608823
   0.00335922  0.00813008 -0.00236935  0.00570383]
 [ 0.0061376  -0.00677874  0.006825   -0.00067857 -0.0020371   0.02585596
  -0.01193887 -0.00805542  0.00135347 -0.00810986]]

x shape :  (3, 10)

e_x 
 [[0.0002 ]
 [0.00549]
 [0.00026]]

cv_x 
 [[4.0e-05 1.0e-05 2.0e-05]
 [1.0e-05 7.0e-05 1.0e-05]
 [2.0e-05 1.0e-05 1.2e-04]]


2. Principal-component model in a 3-stock market: eigenvalues and eigenvectors

In [17]:
# eigenvectors and eigenvalues
# eigh always returns eigenvalues in ascending order
lam, e = eigh(cv_x)

# reversing-resorting eigenvalues
lam = lam[::-1]
print('lam = ', lam.round(5))
print()

# sorted eigenvectors
e = e[:, ::-1]
# flip each eigenvector so its entries sum positive
e = e* ((e.sum(axis=0) >= 0) * 2 - 1)
print('e \n', e.round(2))

lam =  [1.3e-04 7.0e-05 3.0e-05]

e 
 [[ 0.26  0.22  0.94]
 [ 0.26  0.92 -0.29]
 [ 0.93 -0.32 -0.18]]


3. Principal-component model in a 3-stock market: optimal

Loadings and construction matrix

In [18]:
# optimal loadings
beta = e[:, :1]
print('beta \n', beta.round(2))
print()

# Factor construction matrix
gamma = e[:, :1].T
print('gamma = ', gamma.round(2))

beta 
 [[0.26]
 [0.26]
 [0.93]]

gamma =  [[0.26 0.26 0.93]]


4. Principal-component model in a 3-stock market: first principal component

The first principal factor

In [19]:
# expectation of first factor
e_z_pc = gamma@(e_x - e_x)
print('e_z_pc = ', e_z_pc)
print()

# variance of first factor
var_z_pc = gamma@cv_x@gamma.T
print('var_z_pc = ', var_z_pc)
print()

# time series of target variables
print('x_1_t = ', x[0, :].round(4))
print()
print('x_2_t = ', x[1, :].round(4))
print()
print('x_3_t = ', x[2, :].round(4))
print()

e_z_pc =  [[0.]]

var_z_pc =  [[0.00012863]]

x_1_t =  [ 0.0006 -0.0065 -0.0023 -0.0045  0.0064  0.007  -0.0078  0.0113 -0.0041
  0.0018]

x_2_t =  [ 0.0055  0.0213 -0.0034 -0.0063  0.0069  0.0161  0.0034  0.0081 -0.0024
  0.0057]

x_3_t =  [ 0.0061 -0.0068  0.0068 -0.0007 -0.002   0.0259 -0.0119 -0.0081  0.0014
 -0.0081]



5. Principal-component model ina 3-stock market: prediction